<a href="https://www.kaggle.com/code/danny9907/mario-bros?scriptVersionId=248207730" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install gym-super-mario-bros

     |████████████████████████████████| 199 kB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 77 kB 5.5 MB/s  eta 0:00:01
     |████████████████████████████████| 78 kB 6.7 MB/s  eta 0:00:01
  Created wheel for nes-py: filename=nes_py-8.2.1-cp37-cp37m-linux_x86_64.whl size=433871 sha256=47baec6a0844d1b27bfa8243bdf057ec19fac69b27d642357abd3c81a7b8fb28
  Stored in directory: /root/.cache/pip/wheels/17/96/0e/22a8c7dbdf412d8e988286f223b223baf0f4ad90c9e699c56d
Successfully built nes-py
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.45.0
    Uninstalling tqdm-4.45.0:
      Successfully uninstalled tqdm-4.45.0
You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [2]:
import torch
from torch import nn
from torchvision import transforms as T

import time, datetime
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
from collections import deque
import random, datetime, os, copy

import gym
from gym.spaces import Box
from gym.wrappers import FrameStack

from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

# change directory
import os
os.chdir('/kaggle/input/gym-super-mario-bros')

# install controller
!pip --disable-pip-version-check install -q nes_py
from nes_py.wrappers import JoypadSpace

import gym_super_mario_bros

# Initialize Environment

In [3]:
# Initialize Super Mario environment
env = gym_super_mario_bros.make("SuperMarioBros-1-1-v0")

env = JoypadSpace(env, SIMPLE_MOVEMENT)

env.reset()
next_state, reward, done, info = env.step(action=0)
print(f"{next_state.shape},\n {reward},\n {done},\n {info}")

(240, 256, 3),
 0.0,
 False,
 {'coins': 0, 'flag_get': False, 'life': 2, 'score': 0, 'stage': 1, 'status': 'small', 'time': 400, 'world': 1, 'x_pos': 40, 'y_pos': 79}


# Preprocess Environment

In [4]:
class SkipFrame(gym.Wrapper):
    '''
        Custom wrapper that inherits from gy.Wrapper and implements the step() function.
        Use it to return only every skip nth frame
    '''
    def __init__(self, env, skip):
        super().__init__(env)
        self._skip = skip
        
    def step(self, action):
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info

In [5]:
class GrayScaleObservation(gym.ObservationWrapper):
    '''
        Common wrapper to transform an RGB image to grayscale; doing so reduces the size of the state representation without losing useful information.
        Now the size of each state: [1, 240, 256]
    '''
    def __init__(self, env):
        super().__init__(env)
        obs_shape = self.observation_space.shape[:2]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)
        self.transform = T.Grayscale()
    
    def permute_orientation(self, observation):
        observation = np.transpose(observation, (2, 0, 1))
        return torch.tensor(observation.copy(), dtype=torch.float)
    
    def observation(self, observation):
        observation = self.permute_orientation(observation)
        return self.transform(observation)

In [6]:
class ResizeObservation(gym.ObservationWrapper):
    '''
        Downsamples each observation into a square image.
        New size: [1, 84, 84]
    '''
    def __init__(self, env, shape):
        super().__init__(env)
        if isinstance(shape, int): self.shape = (shape, shape)
        else: self.shape = tuple(shape)

        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)
        
    def observation(self, observation):
        transforms = T.Compose([T.Resize(self.shape), T.Normalize(0, 255)])
        return transforms(observation).squeeze(0)

In [7]:
# preprocess environment
env = SkipFrame(env, skip=4)
env = GrayScaleObservation(env)
env = ResizeObservation(env, shape=84)
env = FrameStack(env, num_stack=4)

# Double Deep Q-Networks

In [8]:
class MarioNet(nn.Module):
    """
        input -> (conv2d + relu) x 3 -> flatten -> (dense + relu) x 2 -> output
    """
    def __init__(self, input_dim, output_dim):
        super().__init__()
        c, w, h = input_dim
        
        if h != 84: raise ValueError(f"Expecting input height: 84, got: {h}")
        if w != 84: raise ValueError(f"Expecting input width: 84, got: {w}")
            
        self.online = nn.Sequential(
            nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim)
        )
        
        self.target = copy.deepcopy(self.online)
        
        # freeze Q-target parameters
        for p in self.target.parameters():
            p.requires_grad = False
            
    def forward(self, input, model):
        if model == "online":
            return self.online(input)
        elif model == "target":
            return self.target(input)

# Agent

In [9]:
class Mario:
    '''
        Mario randomly explores with a chance of self.exploration_rate.
        When he chooses to exploit, he relies on MarioNet to provide the most optimal action.
    '''
    def __init__(self, state_dim, action_dim, use_cuda):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.use_cuda = use_cuda
        self.memory = deque(maxlen=20000)
        self.batch_size = 16
        
        self.net = MarioNet(self.state_dim, self.action_dim).float()
        if self.use_cuda:
            self.net = self.net.to(device="cuda")
        
        self.exploration_rate = 1
        self.exploration_rate_decay = 0.99999975
        self.exploration_rate_min = 0.1
        self.curr_step = 0

        self.save_every = 5e5
    
    
    def act(self, state):
        '''
            Given a state, choose an epsilon-greedy action and update value of step.
        '''
        # Exploration
        if np.random.rand() < self.exploration_rate:
            action_idx = np.random.randint(self.action_dim)
        
        # Exploitation
        else:
            state = torch.tensor(state.__array__(), device=self.device).unsqueeze(0)
            action_values = self.net(state, model="online")
            action_idx = torch.argmax(action_values, axis=1).item()
        
        # Decrease exploration_rate
        self.exploration_rate *= self.exploration_rate_decay
        self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)
        
        # Increment step
        self.curr_step += 1
        return action_idx
    
    
    def cache(self, state, next_state, action, reward, done):
        """
            Store the experience to self.memory (replay buffer).
        """
        # Convertir a numpy uint8 comprimido
        state = (state.__array__() * 255).astype(np.uint8)
        next_state = (next_state.__array__() * 255).astype(np.uint8)
    
        # Guardar solo números puros
        self.memory.append((state, next_state, int(action), float(reward), bool(done)))

        
    def recall(self):
        batch = random.sample(self.memory, self.batch_size)
        states, next_states, actions, rewards, dones = zip(*batch)
    
        # Reconstruir tensores desde numpy y normalizarlos
        state = torch.tensor(np.array(states) / 255, dtype=torch.float32)
        next_state = torch.tensor(np.array(next_states) / 255, dtype=torch.float32)
        action = torch.tensor(actions)
        reward = torch.tensor(rewards)
        done = torch.tensor(dones)
    
        if self.use_cuda:
            state = state.cuda()
            next_state = next_state.cuda()
            action = action.cuda()
            reward = reward.cuda()
            done = done.cuda()
    
        return state, next_state, action, reward, done

# TD Estimate & TD Target

In [10]:
class Mario(Mario):
    def __init__(self, state_dim, action_dim, use_cuda):
        super().__init__(state_dim, action_dim, use_cuda)
        self.device = torch.device("cuda" if use_cuda else "cpu")
        self.net = self.net.to(self.device)
        self.gamma = 0.9
        self.burnin = 1e4
        self.learn_every = 6
        self.sync_every = 1e4
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
        self.loss_fn = torch.nn.SmoothL1Loss()


        
    def td_estimate(self, state, action):
        current_Q = self.net(state, model="online")[
            np.arange(0, self.batch_size), action
        ]
        return current_Q

    
    @torch.no_grad()
    def td_target(self, reward, next_state, done):
        next_state_Q = self.net(next_state, model="online")
        best_action = torch.argmax(next_state_Q, axis=1)
        next_Q = self.net(next_state, model="target")[
            np.arange(0, self.batch_size), best_action
        ]
        return (reward + (1 - done.float()) * self.gamma * next_Q).float()
    
    
    def update_Q_online(self, td_estimate, td_target):
        loss = self.loss_fn(td_estimate, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    
    def sync_Q_target(self):
        self.net.target.load_state_dict(self.net.online.state_dict())
        
    
    def learn(self):
        if self.curr_step % self.sync_every == 0: self.sync_Q_target()
        #if self.curr_step % self.save_every == 0: self.save()
        if self.curr_step < self.burnin: return None, None
        if self.curr_step % self.learn_every != 0: return None, None

        # Sample from memory
        state, next_state, action, reward, done = self.recall()
        
        # Mover a GPU
        state = state.to(self.device)
        next_state = next_state.to(self.device)
        action = action.to(self.device)
        reward = reward.to(self.device)
        done = done.to(self.device)
        
        # Get TD Estimate
        td_est = self.td_estimate(state, action)
        
        # Get TD Target
        td_tgt = self.td_target(reward, next_state, done)
        
        # Backpropagate loss through Q_online
        loss = self.update_Q_online(td_est, td_tgt)
        
        return (td_est.mean().item(), loss)

# Logging

In [11]:
class MetricLogger:
    def __init__(self):
        # history metrics
        self.ep_rewards = []
        self.ep_lengths = []
        self.ep_avg_losses = []
        self.ep_avg_qs = []

        # moving averages, added for every call to record()
        self.moving_avg_ep_rewards = []
        self.moving_avg_ep_lengths = []
        self.moving_avg_ep_avg_losses = []
        self.moving_avg_ep_avg_qs = []

        # current episode metric
        self.init_episode()

        # timing
        self.record_time = time.time()

    def log_step(self, reward, loss, q):
        self.curr_ep_reward += reward
        self.curr_ep_length += 1
        if loss:
            self.curr_ep_loss += loss
            self.curr_ep_q += q
            self.curr_ep_loss_length += 1

    def log_episode(self):
        self.ep_rewards.append(self.curr_ep_reward)
        self.ep_lengths.append(self.curr_ep_length)
        if self.curr_ep_loss_length == 0:
            ep_avg_loss = 0
            ep_avg_q = 0
        else:
            ep_avg_loss = np.round(self.curr_ep_loss / self.curr_ep_loss_length, 5)
            ep_avg_q = np.round(self.curr_ep_q / self.curr_ep_loss_length, 5)
        self.ep_avg_losses.append(ep_avg_loss)
        self.ep_avg_qs.append(ep_avg_q)

        self.init_episode()

    def init_episode(self):
        self.curr_ep_reward = 0.0
        self.curr_ep_length = 0
        self.curr_ep_loss = 0.0
        self.curr_ep_q = 0.0
        self.curr_ep_loss_length = 0

    def record(self, episode, epsilon, step):
        mean_ep_reward = np.round(np.mean(self.ep_rewards[-100:]), 3)
        mean_ep_length = np.round(np.mean(self.ep_lengths[-100:]), 3)
        mean_ep_loss = np.round(np.mean(self.ep_avg_losses[-100:]), 3)
        mean_ep_q = np.round(np.mean(self.ep_avg_qs[-100:]), 3)
        self.moving_avg_ep_rewards.append(mean_ep_reward)
        self.moving_avg_ep_lengths.append(mean_ep_length)
        self.moving_avg_ep_avg_losses.append(mean_ep_loss)
        self.moving_avg_ep_avg_qs.append(mean_ep_q)

        last_record_time = self.record_time
        self.record_time = time.time()
        time_since_last_record = np.round(self.record_time - last_record_time, 3)

        print(
            "Episode:{:4d}  :: Step:{:5d}  :: Epsilon:{:8.3f}  :: Mean_Reward:{:8.3f}  :: " \
            "Mean_Length:{:8.3f}  :: Mean_Loss:{:4.3f}  :: Mean_Q_Value:{:8.3f}  :: " \
            "Time_Delta:{:8.3f} "
            .format(episode, step, epsilon, mean_ep_reward, mean_ep_length, 
                    mean_ep_loss, mean_ep_q, time_since_last_record)
        )

# Training

In [12]:
# import gc
# use_cuda = torch.cuda.is_available()
# print(f"Using CUDA: {use_cuda}")
# print()
#
# mario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n, use_cuda=use_cuda)
#
# logger = MetricLogger()
#
# best_mean_reward = -float('inf')  # Inicializamos el mejor reward con un valor muy bajo
#
# episodes = 2001
# for e in range(episodes):
#
#     state = env.reset()
#
#     # Play the game!
#     while True:
#
#         # Run agent on the state
#         action = mario.act(state)
#
#         # Agent performs action
#         next_state, reward, done, info = env.step(action)
#
#         # Remember
#         mario.cache(state, next_state, action, reward, done)
#
#         # Learn
#         q, loss = mario.learn()
#
#         # Logging
#         logger.log_step(reward, loss, q)
#
#         # Update state
#         state = next_state
#
#         # Check if end of game
#         if done or info["flag_get"]:
#             break
#
#     logger.log_episode()
#
#     if e % 25 == 0:
#         logger.record(episode=e, epsilon=mario.exploration_rate, step=mario.curr_step)
#
#         # Evaluar recompensa promedio de los últimos 100 episodios
#         mean_reward = np.mean(logger.ep_rewards[-100:])
#
#         # Si es el mejor modelo hasta ahora, lo guardamos
#         if mean_reward > best_mean_reward:
#             best_mean_reward = mean_reward
#             torch.save(mario.net.state_dict(), '/kaggle/working/best_mario_model.pth')
#             print(f"✅ Modelo guardado en episodio {e} con reward promedio {best_mean_reward:.2f}")
#
#         torch.cuda.empty_cache()
#         gc.collect()


In [13]:
import torch

# Crear una nueva instancia del agente
mario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n, use_cuda=torch.cuda.is_available())

# Cargar el modelo entrenado
mario.net.load_state_dict(torch.load('/kaggle/input/model-trained/best_mario_model (1).pth'))
mario.net.eval()

print("✅ Modelo cargado correctamente. Puedes continuar desde aquí.")

✅ Modelo cargado correctamente. Puedes continuar desde aquí.


In [14]:
!pip --disable-pip-version-check install -q pyvirtualdisplay

from pyvirtualdisplay import Display

from IPython import display as ipythondisplay
from IPython.display import HTML

from gym.wrappers import Monitor
from glob import glob

import base64
import io

display = Display(visible=0, size=(600, 300))
display.start()

In [15]:
def show_video():
    mp4list = glob('/kaggle/working/video/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                 </video>'''.format(encoded.decode('ascii'))))
    else: 
        print("Could not find video")
    

def wrap_env(env):
    env = Monitor(env, '/kaggle/working/video', force=True)
    return env

env = wrap_env(env)

# Play a game

In [16]:
# Para el modelo entrenado
env = wrap_env(env)  # Esto graba en /kaggle/working/video por defecto
state = env.reset()

while True:
    action = mario.act(state)
    next_state, reward, done, info = env.step(action)
    state = next_state
    if done or info["flag_get"]:
        break

env.close()
print("✅ Video del modelo entrenado grabado.")
print("🎮 Mostrando modelo entrenado:")
show_video()

# Para el modelo aleatorio, crear una carpeta diferente
from gym.wrappers import Monitor

def wrap_env_random(env):
    env = Monitor(env, '/kaggle/working/video_random', force=True)
    return env

# Crear entorno como en entrenamiento
rand_env = gym_super_mario_bros.make("SuperMarioBros-1-1-v0")
rand_env = JoypadSpace(rand_env, SIMPLE_MOVEMENT)
rand_env = SkipFrame(rand_env, skip=4)
rand_env = GrayScaleObservation(rand_env)
rand_env = ResizeObservation(rand_env, shape=84)
rand_env = FrameStack(rand_env, num_stack=4)

rand_env = wrap_env_random(rand_env)

state = rand_env.reset()

while True:
    action = rand_env.action_space.sample()
    next_state, reward, done, info = rand_env.step(action)
    state = next_state
    if done or info["flag_get"]:
        break

rand_env.close()
print("✅ Video del modelo aleatorio grabado.")
print("🎮 Mostrando modelo aleatorio:")

# Mostrar el video aleatorio
def show_video_random():
    mp4list = glob('/kaggle/working/video_random/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                 </video>'''.format(encoded.decode('ascii'))))
    else: 
        print("Could not find video")

show_video_random()


✅ Video del modelo entrenado grabado.
🎮 Mostrando modelo entrenado:


✅ Video del modelo aleatorio grabado.
🎮 Mostrando modelo aleatorio:
